# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Dataset description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We'll enumerate all available record sets, their `@id`, and their fields/columns, each referenced by their `@id`.

In [ ]:
# List available record sets with their @id and field/column @ids
print("Available Record Sets in the dataset:")
recordsets = metadata.record_sets
for rs in recordsets:
    print(f"\nRecordSet Name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    print("Fields (@id and name):")
    for field in rs.fields:
        print(f"\t{field.id} -- {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, extract data for *each record set* by their `@id`, so you can refer to them by variable and later choose which to analyze.

In [ ]:
# Extract data from each record set using their @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"RecordSet @id: {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")

# For demonstration, pick the first record set
main_recordset_id = record_set_ids[0]
print(f"\nMain record set to analyze: {main_recordset_id}")
print("Available columns (field @ids):")
print(dataframes[main_recordset_id].columns.tolist())

# Show a preview of the first entries
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following steps apply EDA to a selected numeric field and demonstrate referencing all data elements by their `@id`.

In [ ]:
# List possible numeric fields for the selected record set
rs = next(rs for rs in metadata.record_sets if rs.id == main_recordset_id)
numeric_fields = [field.id for field in rs.fields if field.data_type in ["Float", "Integer", "Number"]]

print("Numeric field @ids available:")
for nf in numeric_fields:
    print(f"  - {nf}")

# Select a field for demo (using the first numeric field):
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    raise ValueError("No numeric field found in the selected record set.")

# Ensure the field is in the dataframe and convertible to numeric
df = dataframes[main_recordset_id]
if numeric_field_id in df.columns:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter for values above a threshold
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If the record set has categorical fields, group by the first one
    categorical_fields = [field.id for field in rs.fields if field.data_type == "Text" and field.id != numeric_field_id]
    group_field_id = categorical_fields[0] if categorical_fields else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by categorical field {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot a histogram of the selected numeric field, and a boxplot grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group if available
if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've used the `mlcroissant` library to explore a Croissant-based dataset. We programmatically loaded available record sets and fields using their unique `@id` references, selected a numeric field for EDA, filtered and normalized values, and visualized the results.

- All references to data fields, record sets, and entities used their `@id` values for clarity and reproducibility.
- This workflow can be adapted to any Croissant dataset for initial review and processing in data science and FAIR data projects.